## Bonus Assignment

On the dataset we used in the lab session (for detecting diabetes), achieve an F1 score above 0.9 (on the test set) using whatever classification method you like (test set should be 20% of the whole data).

**Import libraries**

In [22]:
from sklearn.model_selection import GridSearchCV,train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# read dataset

In [24]:
diabets = pd.read_csv('/content/drive/MyDrive/teias/ads/diabetes.csv')

## Data Preprocessing

In [25]:
diabets.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [26]:
diabets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


**Missing Observation Analysis**

# Invalid data¶
Based on the sample data, Plasma glucose levels, blood pressure, skin thickness, insulin and BMI all have min values at 0 which does not make scence, especially blood pressure, since the diastolic blood pressure most likely cannot be 0 for a living person. Therefore, I believe they have invalid data or missing values.

# We can handle them with :
* **Ignore/remove these casses** - this may not work in the Skin Thickness and Insulin levels have large number of such invalid data points, removing those would leave me with very little data in an already small dataset. This may work for bmi, glucose and BP invalid data points.

* **put median/mean values** - We can fill these missing values with mean or median, which we use median because it is not sensetive to bias and outliers.

* **not using those features for the classification algorthm** - this may work, perhaps in the case of skin thickness.


In [27]:
diabets[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']] = diabets[['Glucose','BloodPressure','SkinThickness','Insulin','BMI']].replace(0,np.NaN)

In [28]:
# Now, we can look at where are missing values
diabets.isnull().sum()

,0
Pregnancies,0
Glucose,5
BloodPressure,35
SkinThickness,227
Insulin,374
BMI,11
DiabetesPedigreeFunction,0
Age,0
Outcome,0


In [29]:
# prompt: fill  0 for these columns ['Glucose','BloodPressure','SkinThickness','Insulin','BMI'] with median according to outcom gruop

# Fill NaN values with the median for each column, grouped by 'Outcome'
for col in ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']:
    diabets[col] = diabets[col].fillna(diabets.groupby('Outcome')[col].transform('median'))

**Split data (Train & Test) & Feature Scaling**

Assuming ```Outcome``` is the column representing the likelihood of a has diabet

In [30]:
diabets.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [35]:
from sklearn.preprocessing import RobustScaler

X = diabets.drop(['Outcome'], axis = 1)
y = diabets['Outcome']
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=23)

scaler = RobustScaler()
x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)


# Prediction

We think randomforest and xgboost are good models to achive f1-score above .9
Also, use gridsearch to finetune models.

In [36]:
from sklearn.model_selection import KFold,cross_val_score

In [40]:
import xgboost as xgb
# Random Forest with Cross-Validation

rf_classifier = RandomForestClassifier(random_state=23)
param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
rf_grid = GridSearchCV(rf_classifier, param_grid_rf, cv=5, scoring='f1')
rf_grid.fit(x_train, y_train)
print("Best Random Forest params:", rf_grid.best_params_)
print("Best Random Forest f1 score (CV):", rf_grid.best_score_)
rf_pred = rf_grid.predict(x_test)
print("Random Forest f1 score (test):", f1_score(y_test, rf_pred))

# XGBoost with Cross-Validation
xgb_classifier = xgb.XGBClassifier(random_state=23, eval_metric='logloss')
param_grid_xgb = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.3]
}
xgb_grid = GridSearchCV(xgb_classifier, param_grid_xgb, cv=5, scoring='f1')
xgb_grid.fit(x_train, y_train)
print("\nBest XGBoost params:", xgb_grid.best_params_)
print("Best XGBoost f1 score (CV):", xgb_grid.best_score_)
xgb_pred = xgb_grid.predict(x_test)
print("XGBoost f1 score (test):", f1_score(y_test, xgb_pred))

Best Random Forest params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Best Random Forest f1 score (CV): 0.8080697411300537
Random Forest f1 score (test): 0.9174311926605505

Best XGBoost params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 50}
Best XGBoost f1 score (CV): 0.818734819957438
XGBoost f1 score (test): 0.8909090909090909


We can see we achive ```f1_score``` above than 90% (91 %) with ```Random Forest Classifier```